In [67]:
from IPython.display import display, Markdown, Latex, JSON

import pandas as pd
import re
from itables import show as itable
import subprocess
from os import path
import pickle
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

## Notes

Notes on HIV and commonly used abbreviations in the context of HIV studies

### Drug Classes

* PI - Protease Inhibitors
* PE - Pharmacokinetic Enhancers
* NNRTI - Non-Nucleoside Reverse Transcriptase Inhibitors
* NRTI - Nucleoside Reverse Transcriptase Inhibitors
* II - Integrase Inhibitors
* E/AI - Entrance/Attachments Inhibitors


### HIV-1 Genome

[![HXB2 HIV-1 Reference Genome](https://www.hiv.lanl.gov/content/sequence/HIV/IMAGES/hxb2genome.gif)](https://www.hiv.lanl.gov/content/sequence/HIV/MAP/landmark.html)


# Importing Resistance Scores and Updating Positions

The resistance scores were obtained from the Stanford HIVdb at the [Drug Resistance Mutations (DRMs) and Sequence Interpretation](https://hivdb.stanford.edu/page/release-notes/#drug.resistance.mutations.drms.and.sequence.interpretation) page. The scores here estimate the level of resistance to a given drug. The estimations of scores are as follows (obtained from [the official documentation of the HIVdb](https://hivdb.stanford.edu/page/release-notes/#drm.penalty.scores.and.resistance.interpretation)):

* Susceptible are values from 0 to 9 (no resistance)
* Potential low-level resistance from 10 to 14
* Low-level resistance from 15 to 29
* Intermediate resistance from 30 to 59
* High level resistance 60 or above.

These scores are typically used as a sum of a given sequence, however, we will be using the scores as an indication of how much it may contribute to drug resistance.

Further more, we will convert the mutation coordinates to those of the HIV-1 HXB2 reference genome.

In [2]:
drug_class_targets = { # The protein targets of the accounted for drugs
    "INSTI": "Integrase",
    "NNRTI": "RT",
    "NRTI": "RT",
    "PI": "Protease"
}

In [81]:
drm_scores = {}
combined_drm_scores = {}

def classify_resistance(score):
    if score < 15:
        return "None"
    elif score < 30:
        return "Low"
    elif score < 60:
        return "Intermediate"
    else:
        return "High"

for drug_classification in drug_class_targets.keys():
    drm_scores[drug_classification] = pd.read_csv("./Data/" + f"{drug_classification} Resistance Mutation Scores.csv")
    combined_drm_scores[drug_classification] = pd.read_csv("./Data/" + f"{drug_classification} Resistance Combined Mutation Scores.csv")


mutation_regex = r"([a-zA-Z]+)(\d+)([a-zA-Z]+)"
for drug_class, drug_scores in drm_scores.items():
    def cbaas_to_hxb2(mut: str):
        match = re.search(mutation_regex, mut)
        pos = int(match.group(2))
        rule = (match.group(1), pos, match.group(3)) # Create mutation string again
        return rule
    # print(.shape)
    drug_scores["Rule"] = drug_scores["Rule"].map(cbaas_to_hxb2)
    drug_scores.insert(1, "Drug Classification", drug_class)
    drug_scores.insert(2, "Score Max", drug_scores.max(axis=1, numeric_only=True))
    drug_scores.insert(3, "Max Resistance", drug_scores["Score Max"].map(lambda v: classify_resistance(float(v))))
    drug_scores.insert(5, "Protein", drug_class_targets[drug_class])

for drug_class, drug_scores in combined_drm_scores.items():
    def cbaas_to_hxb2(mut: str):
        match = re.search(mutation_regex, mut)
        pos = int(match.group(2))
        rule = (match.group(1), pos, match.group(3)) # Create mutation string again
        return rule
    def combined_cbaas_to_hxb2(combined_muts: str):
        results = []
        for mut in combined_muts.split(" + "):
            results.append(cbaas_to_hxb2(mut))
        return results
    drug_scores["Combination Rule"] = drug_scores["Combination Rule"].map(combined_cbaas_to_hxb2)
    drug_scores.insert(2, "Drug Classification", drug_class)
    drug_scores.insert(3, "Score Max", drug_scores.max(axis=1, numeric_only=True))
    drug_scores.insert(4, "Max Resistance", drug_scores["Score Max"].map(lambda v: classify_resistance(float(v))))
    drug_scores.insert(6, "Protein", drug_class_targets[drug_class])

display(Markdown(f"## DRM Scores"))
master_drm_table = pd.concat(drm_scores.values())
itable(master_drm_table)

master_combined_drm_table = pd.concat(combined_drm_scores.values())
display(Markdown(f"## Combination DRM Scores"))
itable(master_combined_drm_table)

## DRM Scores

## Combination DRM Scores

# Find all Mutations Within Epitopes

## Importing Known Epitopes

These epitopes were downloaded from the LANL HIV epitope database using the "CTL/CD8+ Search" search tool with "all" for all parameters or blank for non categorical parameters except:

- Species was set to "Human"

This search was then ran on each of the proteins necessary.

In [44]:
epitope_proteins = ["Integrase", "p24", "Protease", "RT"]
epitope_drug_target_proteins = {
    "Integrase": "Integrase",
    "p24": "Capsid",
    "Protease": "Protease",
    "RT": "RT"
}
protein_epitopes = {}

def junction_resolve(data):
    epitope, location = data
    if "+" not in epitope:
        return epitope
    l_epitope, r_epitope = epitope.split("+")

for epitope_protein in epitope_proteins:
    epitopes = pd.read_csv(f"./Data/Human HIV CTL {epitope_protein} Epitopes.csv")
    epitopes = epitopes.drop( # Removes large, un-needed columns
        labels = [
            "Epitope name",
            "Author location",
            "Vaccine strain",
            "Vaccine component",
            "Vaccine adjuvant",
            "Keywords",
            "Reference",
            "Patient HLA",
            "Species" # We know all data sampled from Humans
        ], 
        axis="columns"
    )
    epitopes = epitopes.dropna(subset = ["Epitope"])
    # epitopes["Epitope"] = epitopes["Epitope"].map() # Synthetic epitopes treated as normal epitopes.
    epitopes = epitopes.drop_duplicates(subset="Epitope")
    protein_epitopes[epitope_protein] = epitopes

for epitope_protein, epitopes in protein_epitopes.items():
    display(Markdown(f"### {epitope_protein} Epitopes"))
    itable(epitopes)

### Integrase Epitopes

### p24 Epitopes

### Protease Epitopes

### RT Epitopes

## Generating Mutated Epitopes

We will generate the mutated epitopes with the reference sequence being HXB2. To do this, we will iterate over each epitope, and check a hashmap for the mutations at each position in each epitopes. An upper limit on the length of the epitopes is usually ~15 residues. Therefore, the average running time will be $\Theta(n)$ where $n$ is the number of epitopes used.

In [5]:
# Generate hashmap of locations and their mutations
epitope_mutation_map = {}
for drug_class, drm_score in drm_scores.items():
    if drug_class not in epitope_mutation_map:
        drug_target = drug_class_targets[drug_class]
        epitope_mutation_map[drug_target] = {}
    for index, hxb2_rule in drm_score[["Mutation Rule"]].itertuples():
        original, position, mutation = hxb2_rule
        position = int(position)
        if position not in epitope_mutation_map[drug_target]:
            epitope_mutation_map[drug_target][position] = list()
        epitope_mutation_map[drug_target][position].append(hxb2_rule)
display(JSON(epitope_mutation_map))

<IPython.core.display.JSON object>

In [38]:
# Determine binding level function
def binding_level(el_rank: float):
    if el_rank < 0.5:
        return "Strong"
    elif el_rank < 2:
        return "Weak"
    else:
        return "None"

# netMHCpan Interface Function
def predict_antigenicity(seqs: list[str], hla_alleles: list[str], print_output = False):
    temp = "Temporary"
    peptide_path = path.join(temp, "peptide")
    xls_path = path.join(temp, "output.tsv")
    with open(peptide_path, "w") as peptide_file:
        peptide_file.writelines(s + '\n' for s in seqs)
    
    total_results = []
    for hla_allele in hla_alleles:
        process_out = subprocess.run(["conda", "run", "-p", "./env", "./Software/netMHCpan-4.1/netMHCpan",
                        "-BA", "-xls",
                        "-a", hla_allele,
                        "-p", peptide_path,
                        "-xlsfile", xls_path
                    ],
                    capture_output=True
                    )
        if print_output:
            print(process_out)
        result_table = pd.read_csv(xls_path, sep="\t")
        result_table.columns = result_table.iloc[0]
        result_table = result_table.drop(0, axis = 0)
        result_table = result_table.drop(
            [
                "Pos",
                "ID",
                "Ave",
                "NB",
                "icore",
                "core",

            ],
            axis=1
        )
        result_table.insert(loc=1, column="HLA Allele", value=hla_allele)
        
        result_table.insert(loc=2, column="Binding Level", value=result_table["EL_Rank"].map(lambda v: binding_level(float(v))))
        total_results.append(result_table)
    return pd.concat(total_results, ignore_index=True)

test_netMHCpan_results = predict_antigenicity(['GQVDCSPGIWQLDCTH'], hla_alleles=["HLA-A02:01", "HLA-A01:01"])
itable(test_netMHCpan_results)

# Predicting Antigenicity and Binding Affinity


In [59]:

epitope_protein_region_alias = {
    "Integrase": "Pol",
    "p24": "Gag",
    "Protease": "Pol",
    "RT": "Pol"
}

all_hla_alleles = [
    "HLA-A02:01",
    "HLA-A01:01",
    "HLA-A03:01",
    "HLA-A24:02",
    "HLA-A26:01",
    "HLA-B07:02",
    "HLA-B08:01",
    "HLA-B27:05",
    "HLA-B39:01",
    "HLA-B40:01",
    "HLA-B58:01",
    "HLA-B15:01"
]

# TODO PDB has a common HLA allele list for epitope prediction software
# It's a default HIV panel.
common_hla_alleles = [
    "HLA-A01:01",
    "HLA-A02:01",
    "HLA-A03:01",
    # Missing HLA-A*11:01
    "HLA-A24:02"
]

disease_associated_hla_alleles = [
    "HLA-A01:01",
    "HLA-A02:01",
    "HLA-A03:01"
    # Missing HLA-A*11:01
]

hiv_associated_hla_alleles = [
    # Missing HLA-B*57 (generates considerable immune response)
    # Missing other HLA-B27, HLA-B57 (typically protective)
    "HLA-B27:05", # typically protective
    "HLA-B58:01", # typically protective
]

### Configuration
hla_alleles = common_hla_alleles
max_per_epitope = 100

In [60]:

### Running
antigenicity_table = []
antigenicity_cache_path = "Cache/antigenicity.pickle"
if path.isfile(antigenicity_cache_path):
    print("Loading from cache...")
    with open(antigenicity_cache_path, 'rb') as handle:
        antigenicity_table = pickle.load(handle)
else:
    process_log = []
    # Generate mutated epitopes
    unmutated_epitopes_to_test = []
    for epitope_protein in tqdm(epitope_proteins, desc="Epitope Protein Origin"):
        epitopes_table = protein_epitopes[epitope_protein]
        drug_target = epitope_drug_target_proteins[epitope_protein]
        region_alias = epitope_protein_region_alias[epitope_protein]
        if drug_target not in epitope_mutation_map:
            process_log.append(f"Skipping \"{drug_target}\".")
            continue
        regex = rf"{region_alias}\((\d+)-(\d+)\)"
        epitopes = list(epitopes_table[["Epitope", "HXB2 location"]].itertuples())
        epitopes = epitopes[:min(max_per_epitope, len(epitopes))] if max_per_epitope != 0 else epitopes
        for index, epitope, location_str in tqdm(epitopes, desc=f"{epitope_protein}"):
            location_match = re.search(regex, location_str)
            start = int(location_match.group(1))
            end = int(location_match.group(2))
            if (end - start + 1) != len(epitope):
                process_log.append(f"Epitope \"{epitope}\" of {drug_target} has length {len(epitope)}. Database indicates it starts at {start} and ends at {end} with length of {end - start + 1} ({end - start + 1} != {len(epitope)}). It will be skipped.")
                continue
            epitopes_to_test = []
            corresponding_positions = []
            corresponding_peptides = []
            corresponding_mutations = []
            reversed_mutations = []
            unmutated_epitopes_to_test.append(epitope)

            for position in range(start, start + len(epitope)):
                position = position
                mutation_position = position - start
                current_aa = epitope[mutation_position]
                if position in epitope_mutation_map[drug_target]:
                    mutations = epitope_mutation_map[drug_target][position]
                    for original_aa, pos, mutation_aa in epitope_mutation_map[drug_target][position]:
                        mutated_epitope = None
                        if original_aa == current_aa:
                            mutated_epitope = epitope[:mutation_position] + mutation_aa + epitope[mutation_position + 1:]
                            process_log.append(f"{start}-{epitope}-{end} Testing HLA binding affinity on sequence {mutated_epitope}")
                            reversed_mutations.append(False)
                        elif mutation_aa == current_aa:
                            mutated_epitope = epitope[:mutation_position] + "*" + original_aa + "*" + epitope[mutation_position + 1:]
                            process_log.append(f"{start}-{epitope}-{end} Testing HLA binding affinity on sequence {mutated_epitope}")
                            reversed_mutations.append(True)
                        else:
                            process_log.append(f"{start}-{epitope}-{end} Expected {original_aa} or {mutation_aa} at {position} for epitope in {drug_target} ({region_alias}). Found {current_aa} instead.")
                            pass
                        if mutated_epitope is not None:
                            epitopes_to_test.append(mutated_epitope)
                            corresponding_positions.append((start, end))
                            corresponding_peptides.append(epitope)
                            corresponding_mutations.append((original_aa, pos, mutation_aa))
            if len(epitopes_to_test) > 1:
                for hla_allele in hla_alleles:
                    protein_epitope_antigenicities = predict_antigenicity(epitopes_to_test, hla_alleles=[hla_allele])
                    protein_epitope_antigenicities.insert(loc=1, column="Original Peptide", value=corresponding_peptides)
                    protein_epitope_antigenicities.insert(loc=3, column="Mutation Rule", value=corresponding_mutations)
                    protein_epitope_antigenicities.insert(loc=4, column="Peptide Position", value=corresponding_positions)
                    protein_epitope_antigenicities.insert(loc=5, column="Mutation Reversed", value=reversed_mutations)
                    protein_epitope_antigenicities.insert(loc=6, column="Protein", value=epitope_protein)
                    antigenicity_table.append(protein_epitope_antigenicities)
    antigenicity_table = pd.concat(antigenicity_table)
    with open(antigenicity_cache_path, 'wb') as handle:
        print("Saving to cache...")
        pickle.dump(antigenicity_table, handle)
    unmutated_antigenicity_table = predict_antigenicity(unmutated_epitopes_to_test, hla_alleles=hla_alleles)
    print("\n".join(process_log))

itable(antigenicity_table)
itable(unmutated_antigenicity_table)

Epitope Protein Origin:   0%|          | 0/4 [00:00<?, ?it/s]

Integrase:   0%|          | 0/100 [00:00<?, ?it/s]

Protease:   0%|          | 0/79 [00:00<?, ?it/s]

RT:   0%|          | 0/100 [00:00<?, ?it/s]

Saving to cache...
Epitope "DKAQ+AKEIVASCDKS" of Integrase has length 16. Database indicates it starts at 748 and ends at 758 with length of 11 (11 != 16). It will be skipped.
Epitope "TAYFLLKLA+MNKELK" of Integrase has length 16. Database indicates it starts at 812 and ends at 820 with length of 9 (9 != 16). It will be skipped.
Skipping "Capsid".
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence EITLWQRPLVSIKVGGQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGAQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGCQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGDQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGSQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGTQI
58-QITLWQRPLVSIKVGGQI-75 Testing HLA binding affinity on sequence QITLWQRPLVSIKVGVQI
58-QITLWQRPLVSIKVGGQI-75 Expected T or P at 74 for epit

# Mapping Original Antigenicities and Mutated Antigencities

In [71]:
antigenicity_change_table = pd.merge(
    left=antigenicity_table,
    right=unmutated_antigenicity_table,
    left_on=["Original Peptide", "HLA Allele"],
    right_on=["Peptide", "HLA Allele"],
    suffixes=[" Mutated", " Original"]
)
antigenicity_change_table = antigenicity_change_table.drop(
    [
        "Peptide Original"
    ],
    axis=1
)
antigenicity_change_table.insert(
    loc=6,
    column="Binding Level Change",
    value=antigenicity_change_table[["Binding Level Mutated", "Binding Level Original"]].apply(lambda v: "None" if v.iloc[0] == v.iloc[1] else f"{v.iloc[1]} to {v.iloc[0]}", axis=1)
)
antigenicity_change_table.insert(
    loc=7,
    column="EL Change",
    value=antigenicity_change_table[["EL_Rank Mutated", "EL_Rank Original"]].apply(lambda v: float(v.iloc[1]) - float(v.iloc[0]), axis=1)
)
itable(antigenicity_change_table)

# Mapping Drug Resistance Scores

In [72]:
master_antigenicity_drm_table = pd.merge(
    left=antigenicity_change_table,
    right=master_drm_table,
    how="left",
    left_on=["Mutation Rule", "Protein"],
    right_on=["Mutation Rule", "Protein"]
)

display(Markdown(f"## Master Drug Resistance and Antigenicity Score Table"))
itable(master_antigenicity_drm_table, maxBytes=0)


## Master Drug Resistance and Antigenicity Score Table

# Summarize

## Table

In [77]:
results_summary_table = master_antigenicity_drm_table[["Peptide Mutated", "Original Peptide", "HLA Allele", "Mutation Rule", "Mutation Reversed", "Peptide Position", "Binding Level Change", "Protein", "Max Resistance"]]
results_summary_table = results_summary_table[results_summary_table["Binding Level Change"] != "None"]
results_summary_table = results_summary_table[results_summary_table["Max Resistance"] != "None"]
itable(results_summary_table)